<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/%D0%98%D0%BD%D1%84%D0%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## Лекция 6.2. Масштабирование RAG без фреймворков

**Цель:** превратить игрушечный пример в систему, работающую с реальными документами, векторной БД, историей и интеллектуальной маршрутизацией – всё на чистом Python.

### Тема 1. Загрузка документов из папки
1.1. Поддерживаемые форматы: `.txt`, `.pdf` (PyPDF2/pypdf), `.docx` (python-docx), `.md` (просто текст).  
1.2. Обход папки рекурсивно – `os.walk` или `pathlib`.  
1.3. Извлечение текста из каждого файла с обработкой ошибок (битые PDF, кодировки).  
1.4. Сохранение метаданных: имя файла, путь, дата изменения, номер страницы (для PDF).  
1.5. Сбор всех текстов в единый список с привязкой к метаданным.

### Тема 2. Умное разбиение на чанки
2.1. Понятие чанка: зачем нужен перекрытие (overlap) и ограничение по длине.  
2.2. Реализация рекурсивного сплиттера: сначала по абзацам (`\n\n`), затем по предложениям (`. `, `! `, `? `), затем по словам, чтобы не превышать `chunk_size`.  
2.3. Параметры: `chunk_size` (например, 500 символов), `chunk_overlap` (50–100 символов).  
2.4. Сохранение метаданных для каждого чанка (источник, номер чанка).  
2.5. Удаление пустых чанков и нормализация пробелов.

### Тема 3. Векторная база данных Chroma
3.1. Установка `chromadb`. Запуск в режиме `PersistentClient` (сохранение на диск).  
3.2. Создание коллекции с указанием функции эмбеддингов (можно передавать свои эмбеддинги напрямую).  
3.3. Добавление документов (чанков) в коллекцию: `collection.add(documents=[...], metadatas=[...], ids=[...])`.  
3.4. Поиск: `collection.query(query_texts=[question], n_results=top_k)`.  
3.5. Сравнение с ручным вычислением косинуса из Лекции 1 – почему Chroma удобнее (индексация, масштабируемость, хранение на диске).  
3.6. Обновление базы при добавлении новых документов (или пересборка с нуля).

### Тема 4. Интеллектуальная маршрутизация через LLM (вместо ключевых слов)
4.1. Отказ от списка `topics`. Формирование промпта: попросить модель вернуть JSON с полем `"action"` (`search` или `answer`).  
4.2. Описание контекста: какие темы есть в документах (можно передать краткое описание коллекции).  
4.3. Парсинг ответа: `json.loads()`. Обработка ошибок: если модель вернула невалидный JSON, fallback – всегда `search`.  
4.4. Сравнение точности маршрутизации: на примерах где раньше были ошибки (омонимы, синонимы).  
4.5. Дополнительно: модель может вернуть `confidence` (уверенность), чтобы принимать решение более гибко.

### Тема 5. Краткосрочная память (история диалога)
5.1. Структура: список словарей `[{"role": "user"/"assistant", "content": "..."}]`.  
5.2. Добавление истории в промпт: форматирование как последовательность сообщений.  
5.3. Ограничение длины истории – оставляем последние N обменов (например, 5).  
5.4. Как история влияет на маршрутизацию: если предыдущий вопрос был про документы, следующий скорее всего тоже.  
5.5. Очистка истории по команде (например, `/reset`).

### Тема 6. Логирование и отладка
6.1. Модуль `logging` – настройка уровней (INFO, DEBUG).  
6.2. Запись в лог: вопрос, выбранное действие, найденные чанки (их тексты и оценки), финальный ответ.  
6.3. Вывод в консоль и в файл одновременно.  
6.4. Использование логов для анализа ошибок и улучшения системы.

**Итоговый код**: единый скрипт или класс `RAGAgent`, который принимает папку с документами, строит индекс, и в цикле отвечает на вопросы с историей.

---

## Лекция 6.3. Введение в LangChain

**Цель:** переписать ту же функциональность, но с использованием абстракций LangChain – меньше кода, больше гибкости, подготовка к агентам.

### Тема 1. Что такое LangChain и зачем он нужен
1.1. Основные компоненты: LLM, промпты, цепочки, ретриверы, память.  
1.2. Установка `langchain`, `langchain-community`, `langchain-chroma`, `langchain-ollama`.  
1.3. Обзор архитектуры – как всё связано (LCEL – LangChain Expression Language).

### Тема 2. Вызов LLM через ChatOllama
2.1. Инициализация модели: `ChatOllama(model="qwen2.5:3b")`.  
2.2. Отправка сообщений: `llm.invoke([HumanMessage(content="...")])`.  
2.3. Получение ответа через `content`.  
2.4. Сравнение с прямым HTTP‑запросом из Лекции 1.

### Тема 3. Промпты и парсеры
3.1. `ChatPromptTemplate` – шаблоны с переменными.  
3.2. `SystemMessage`, `HumanMessage`, `AIMessage` – структурирование диалога.  
3.3. `PydanticOutputParser` – описываем структуру JSON через Pydantic, получаем парсинг без `json.loads()` вручную.  
3.4. Пример: парсер для маршрутизации (`{"action": "search"}`).

### Тема 4. Создание цепочки RAG с помощью LCEL
4.1. Загрузка векторной базы из Chroma: `Chroma(persist_directory=..., embedding_function=...)`.  
4.2. Создание ретривера: `vectorstore.as_retriever(search_kwargs={"k": 3})`.  
4.3. Цепочка: `prompt | llm | StrOutputParser()`.  
4.4. Встраивание ретривера через `RunnablePassthrough.assign(context=retriever)`.  
4.5. Сборка полной RAG‑цепочки: вопрос → поиск → формирование промпта → генерация.  
4.6. Сравнение количества строк кода с Лекцией 2.

### Тема 5. Добавление памяти в цепочку
5.1. `ConversationBufferMemory` – хранит историю.  
5.2. Использование `RunnableWithMessageHistory` для автоматической подстановки истории.  
5.3. Интеграция с RAG‑цепочкой: память включает предыдущие вопросы и ответы.

### Тема 6. Парсинг структурированных ответов (для выбора инструмента)
6.1. Использование `PydanticOutputParser` в связке с `ChatPromptTemplate`.  
6.2. Пример: модель возвращает JSON с действием.  
6.3. Обработка ошибок парсинга – fallback.

**Итоговый код**: классная RAG‑цепочка с памятью, которая занимает 30–40 строк вместо 150+ в Лекции 2.

---

## Лекция 6.4. Агент на LangGraph

**Цель:** построить настоящего агента с циклом «думать → действовать → наблюдать», который может последовательно вызывать инструменты и принимать решения на основе промежуточных результатов.

### Тема 1. Зачем нужен LangGraph (вместо простой цепочки)
1.1. Ограничения цепочек: нет ветвления, нет циклов.  
1.2. Понятие графа состояний – узлы (nodes) и рёбра (edges).  
1.3. Состояние – словарь, который передаётся между узлами (например, список сообщений, промежуточные результаты).  
1.4. Установка `langgraph`.

### Тема 2. Проектирование графа агента
2.1. Узлы:  
   - `agent` – принимает решение: ответить или вызвать инструмент.  
   - `tools` – выполняет вызванный инструмент (поиск, калькулятор и т.д.).  
   - `final_answer` – генерирует финальный ответ пользователю.  
2.2. Рёбра:  
   - `agent` → `tools` (если решил вызвать инструмент)  
   - `agent` → `final_answer` (если решил ответить)  
   - `tools` → `agent` (после выполнения инструмента – снова дать модели подумать)  
2.3. Условные рёбра – функция, которая анализирует состояние и определяет следующий узел.

### Тема 3. Инструменты в LangGraph (декоратор @tool)
3.1. Создание функций: `search_docs(query)`, `calculate(expression)`, `get_current_time()`, `web_search(query)` (через DuckDuckGo).  
3.2. Оборачивание в `@tool` – добавляет имя, описание и схему параметров.  
3.3. Передача списка инструментов в LLM через `bind_tools()` – модель автоматически учится их вызывать.  
3.4. Обработка вызова инструмента: извлечение `tool_calls` из ответа модели.

### Тема 4. Реализация цикла «агент-инструменты-агент»
4.1. Узел `agent` вызывает LLM с историей и инструментами.  
4.2. Если есть `tool_calls` – переходим к узлу `tools`.  
4.3. Узел `tools` выполняет каждый вызов, добавляет результат в состояние как новое сообщение.  
4.4. Возврат в узел `agent` (цикл).  
4.5. Ограничение числа итераций – защита от бесконечного цикла.

### Тема 5. Добавление памяти (сохранение состояния между запросами)
5.1. Использование `MemorySaver` – автоматическое сохранение состояния после каждого шага.  
5.2. Передача `thread_id` для разных сессий.  
5.3. Как агент помнит историю диалога и может задавать уточняющие вопросы.

### Тема 6. Human‑in‑the‑loop (прерывания)
6.1. Понятие `interrupt` – остановка выполнения перед определённым узлом.  
6.2. Пример: перед выполнением `web_search` запросить подтверждение у пользователя.  
6.3. Реализация через `graph.compile(checkpointer=memory, interrupt_before=["tools"])`.

**Итоговый код**: класс-агент на LangGraph с несколькими инструментами, памятью и возможностью прерывания.

---

## Лекция 6.5. Доводка до продакшена

**Цель:** углубить качество ответов, добавить мониторинг и упаковать в интерфейс.

### Тема 1. Гибридный поиск и реранкинг
1.1. Проблемы одного семантического поиска: пропуск точных совпадений (ключевые слова).  
1.2. Добавление BM25 (через `rank_bm25`) – поиск по ключевым словам.  
1.3. Объединение результатов: комбинирование оценок (weighted sum или реранкинг).  
1.4. Использование кросс-энкодеров (например, `cross-encoder/ms-marco-MiniLM-L-6-v2`) для переоценки релевантности найденных чанков – улучшение точности на 5–10%.

### Тема 2. Мониторинг и трассировка
2.1. Подключение LangSmith – бесплатный план.  
2.2. Отслеживание каждого шага агента: промпты, ответы, вызовы инструментов.  
2.3. Альтернатива: своё логирование с детализацией (время выполнения, токены).  
2.4. Использование логов для анализа ошибок и доработки промптов.

### Тема 3. Асинхронность и производительность
3.1. Зачем асинхронность: чтобы обрабатывать несколько запросов одновременно (для веб-сервера).  
3.2. Переход на `async` версии методов LangChain (`ainvoke`, `astream`).  
3.3. Потоковая передача ответов (streaming) для улучшения UX.

### Тема 4. Развёртывание в веб‑интерфейсе
4.1. Простое приложение на Streamlit: поле ввода, вывод истории, логи в отдельном окне.  
4.2. Асинхронный бот в Telegram через `python-telegram-bot` – подключение агента как обработчика сообщений.  
4.3. Вариант с FastAPI – создание REST API для агента.

### Тема 5. Безопасность и ограничения
5.1. Санитизация пользовательского ввода – предотвращение prompt-инъекций.  
5.2. Ограничение длины запроса и ответа, таймауты.  
5.3. Управление доступом к инструментам (например, разрешить `web_search` только администраторам).  
5.4. Резервное копирование векторной базы.

### Тема 6. Планирование и самооценка (взгляд в будущее)
6.1. Понятие планирования – модель составляет план действий перед выполнением.  
6.2. Самооценка – модель проверяет свой ответ и при необходимости переспрашивает.  
6.3. Краткий обзор современных подходов: ReAct, Reflexion, Tree of Thoughts.

**Итоговый код**: готовый продукт – агент с расширенным поиском, веб‑интерфейсом или телеграм‑ботом, готовый к эксплуатации.
